# 01_saga_compensate — Saga and compensations

In a distributed system there is no shared transaction. The Saga pattern gives each step a **compensating** operation that undoes its effect; on failure the executed steps are rolled back in **reverse order**. In AOA the compensator is declared with `@compensate` right next to its step — a visible part of the contract, not glue in the code.

This notebook fulfills an order in three steps — reserve inventory, charge payment, confirm — where the final step fails on purpose. The machine unwinds the executed steps backwards: refund, then release.

**What's new**

| Concept | Description |
|---------|-------------|
| `@regular_aspect` + `@result_string` | A step that produces checked state |
| `@compensate(aspect_name, "desc")` | Binds a compensator to a step **by callable reference** |
| compensator signature | `(self, params, state_before, state_after, box, connections, error)` |
| reverse-order unwind | last executed step is compensated first |
| original error surfaces | after rollback, the step's original exception is raised |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import GuestRole
from aoa.action_machine.context import Context
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import regular_aspect, summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.checkers import result_string
from aoa.action_machine.intents.compensate import compensate
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.logging import Channel, ConsoleLogger
from aoa.action_machine.logging.log_coordinator import LogCoordinator
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Params, Result

In [ ]:
class RootDomain(BaseDomain):
    name = "root"
    description = "Root domain"


class OrderParams(BaseParams):
    order_id: str = Field(description="Order ID")


class OrderResult(BaseResult):
    order_id: str = Field(description="Order ID")
    status: str = Field(description="Status")

## The action — each step paired with its compensator

`reserve_aspect` ↔ `reserve_compensate` (release the reservation), `charge_aspect` ↔ `charge_compensate` (refund). The compensator is bound **by direct callable reference**; the target must be defined before the compensator in the class body. The summary step fails deliberately — `confirm_summary` raises — to trigger the unwind.

In [ ]:
@meta(description="Fulfill order (saga)", domain=RootDomain)
@check_roles(GuestRole)
class FulfillOrderAction(BaseAction[OrderParams, OrderResult]):

    @regular_aspect("Reserve inventory")
    @result_string("reservation_id", required=True)
    async def reserve_aspect(self, params, state, box, connections):
        await box.info(
            Channel.business,
            "regular: reserve order={%var.order_id}",
            order_id=params.order_id,
        )
        return {"reservation_id": f"res-{params.order_id}"}

    @compensate(reserve_aspect, "Release reservation")
    async def reserve_compensate(self, params, state_before, state_after, box, connections, error):
        rid = state_after["reservation_id"] if state_after else "?"
        await box.info(
            Channel.business,
            "compensate: release reservation {%var.rid}",
            rid=rid,
        )

    @regular_aspect("Charge payment")
    @result_string("txn_id", required=True)
    async def charge_aspect(self, params, state, box, connections):
        await box.info(
            Channel.business,
            "regular: charge order={%var.order_id}",
            order_id=params.order_id,
        )
        return {"txn_id": f"txn-{params.order_id}"}

    @compensate(charge_aspect, "Refund payment")
    async def charge_compensate(self, params, state_before, state_after, box, connections, error):
        txn = state_after["txn_id"] if state_after else "?"
        await box.info(
            Channel.business,
            "compensate: refund {%var.txn}",
            txn=txn,
        )

    @summary_aspect("Confirm order")
    async def confirm_summary(self, params, state, box, connections):
        raise ValueError("order service unavailable")

## Run

Reserve and charge succeed; the confirm step fails. The unwind runs from the end: `charge_compensate` (refund) first, then `reserve_compensate` (release). Only after the rollback does the original error surface.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine(
        log_coordinator=LogCoordinator(loggers=[ConsoleLogger()])
    )
    try:
        await machine.run(
            Context(),
            FulfillOrderAction(),
            params=OrderParams(order_id="ord-001"),
        )
    except ValueError as e:
        print("\n" + str(e))


await main()